# FACED–MSGM Resume Notebook

এই notebook শুধুমাত্র previous Kaggle run-এর saved output থেকে training resume করার জন্য।

Kaggle notebook-এ input হিসেবে যোগ করুন:

1. Original **MSGM-main** dataset
2. আপনার previous notebook-এর latest output dataset, যেখানে `faced_msgm/` folder আছে
3. Accelerator হিসেবে **GPU T4**

Expected saved structure:

```text
faced_msgm/
├── checkpoints/
├── results/
├── rpsd_subjects/
├── reproduction_config.json
├── subject_folds.json
└── sample_manifest.csv
```

এই notebook completed folds skip করবে। অসম্পূর্ণ Fold 7 checkpoint থাকলে সেটি delete করে Fold 7 epoch 1 থেকে cleanly restart করবে।

In [1]:
# Cell 1: Import libraries and inspect Kaggle inputs

from pathlib import Path
from dataclasses import dataclass, asdict
from types import SimpleNamespace

import os
import sys
import gc
import json
import time
import shutil
import zipfile
import random

import numpy as np
import pandas as pd
import torch

KAGGLE_INPUT = Path("/kaggle/input")
KAGGLE_WORKING = Path("/kaggle/working")

print("Available Kaggle inputs:")
for path in sorted(KAGGLE_INPUT.iterdir()):
    print("-", path)


Available Kaggle inputs:
- /kaggle/input/datasets


In [2]:
# Cell 2: Automatically locate MSGM code and previous faced_msgm output

def find_directory_named(root: Path, name: str):
    matches = [p for p in root.rglob(name) if p.is_dir()]
    return matches


# Locate MSGM-main
msgm_matches = find_directory_named(KAGGLE_INPUT, "MSGM-main")

if not msgm_matches:
    raise FileNotFoundError(
        "MSGM-main was not found. Add the MSGM Kaggle dataset as input."
    )

MSGM_INPUT_ROOT = msgm_matches[0]


# Locate previous faced_msgm folder
faced_output_matches = find_directory_named(KAGGLE_INPUT, "faced_msgm")

# Prefer a folder containing the core resume files.
valid_output_matches = [
    p for p in faced_output_matches
    if (p / "reproduction_config.json").exists()
    and (p / "subject_folds.json").exists()
    and (p / "rpsd_subjects").exists()
]

PREVIOUS_OUTPUT_ROOT = (
    valid_output_matches[0]
    if valid_output_matches
    else None
)

# Also support a packaged ZIP output.
zip_matches = list(
    KAGGLE_INPUT.rglob("faced_msgm_reproduction_outputs.zip")
)

print("MSGM input root:", MSGM_INPUT_ROOT)
print("Previous faced_msgm folder:", PREVIOUS_OUTPUT_ROOT)
print("Possible packaged ZIP files:", zip_matches)

if PREVIOUS_OUTPUT_ROOT is None and not zip_matches:
    raise FileNotFoundError(
        "Previous faced_msgm output was not found. "
        "Add the latest notebook output as a Kaggle input dataset."
    )


MSGM input root: /kaggle/input/datasets/nushratjaben/msgm-gnn/MSGM-main
Previous faced_msgm folder: /kaggle/input/datasets/nushratjaben/msgm-ckpt/faced_msgm
Possible packaged ZIP files: []


In [3]:
# Cell 3: Restore previous output and MSGM code into /kaggle/working

WORK_DIR = KAGGLE_WORKING / "faced_msgm"
LOCAL_MSGM_ROOT = KAGGLE_WORKING / "MSGM-main"

# Restore faced_msgm output.
if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)

if PREVIOUS_OUTPUT_ROOT is not None:
    shutil.copytree(
        PREVIOUS_OUTPUT_ROOT,
        WORK_DIR,
    )
else:
    WORK_DIR.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(zip_matches[0], "r") as archive:
        archive.extractall(WORK_DIR)

    # Handle ZIPs that contain a nested faced_msgm folder.
    nested = WORK_DIR / "faced_msgm"
    if nested.exists() and nested.is_dir():
        temp_dir = KAGGLE_WORKING / "_faced_msgm_temp"
        if temp_dir.exists():
            shutil.rmtree(temp_dir)
        shutil.move(str(nested), str(temp_dir))
        shutil.rmtree(WORK_DIR)
        shutil.move(str(temp_dir), str(WORK_DIR))


# Restore MSGM source.
if LOCAL_MSGM_ROOT.exists():
    shutil.rmtree(LOCAL_MSGM_ROOT)

shutil.copytree(
    MSGM_INPUT_ROOT,
    LOCAL_MSGM_ROOT,
)

FEATURE_DIR = WORK_DIR / "rpsd_subjects"
CHECKPOINT_DIR = WORK_DIR / "checkpoints"
RESULT_DIR = WORK_DIR / "results"

for directory in [
    FEATURE_DIR,
    CHECKPOINT_DIR,
    RESULT_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

print("Restored output:", WORK_DIR)
print("Restored MSGM code:", LOCAL_MSGM_ROOT)
print("Cached subject files:", len(list(FEATURE_DIR.glob("sub*.npz"))))


Restored output: /kaggle/working/faced_msgm
Restored MSGM code: /kaggle/working/MSGM-main
Cached subject files: 123


In [4]:
# Cell 4: Install requirements and import official MSGM package

!pip install -q pyyaml scipy openpyxl scikit-learn

if str(LOCAL_MSGM_ROOT) not in sys.path:
    sys.path.insert(0, str(LOCAL_MSGM_ROOT))

import importlib
import msgm

importlib.reload(msgm)

from msgm import MSGM, count_parameters

print("MSGM imported from:", msgm.__file__)


MSGM imported from: /kaggle/working/MSGM-main/msgm/__init__.py


In [5]:
# Cell 5: Load saved reproduction configuration and subject folds

@dataclass
class ReproductionConfig:
    sampling_rate: int = 250
    num_channels: int = 32
    num_features: int = 7
    num_classes: int = 2

    first_window_seconds: float = 20.0
    first_hop_seconds: float = 4.0
    sub_window_seconds: tuple = (4.0, 3.0, 2.0, 1.0)
    sub_overlap: float = 0.75
    scale_lengths: tuple = (16, 23, 36, 77)

    hidden_dim: int = 32
    graph_layers: tuple = (1, 2)
    chebyshev_order: int = 4
    mamba_layers: int = 1
    mamba_state_dim: int = 16
    mamba_conv_kernel: int = 4
    mamba_expand: int = 2
    dropout: float = 0.25

    learning_rate: float = 3e-4
    weight_decay: float = 1e-2
    label_smoothing: float = 0.1
    batch_size: int = 32
    epochs: int = 30
    early_stopping_patience: int = 5
    random_seed: int = 42

    num_test_folds: int = 10
    fold_mode: str = "paper_close_10x12"
    validation_mode: str = "subject_wise"
    neutral_policy: str = "exclude"


config_path = WORK_DIR / "reproduction_config.json"
fold_path = WORK_DIR / "subject_folds.json"

if not config_path.exists():
    raise FileNotFoundError(config_path)

if not fold_path.exists():
    raise FileNotFoundError(fold_path)

config_data = json.loads(
    config_path.read_text(encoding="utf-8")
)

# JSON stores tuples as lists.
for key in [
    "sub_window_seconds",
    "scale_lengths",
    "graph_layers",
]:
    if key in config_data:
        config_data[key] = tuple(config_data[key])

CONFIG = ReproductionConfig(**config_data)

fold_data = json.loads(
    fold_path.read_text(encoding="utf-8")
)
folds = fold_data["folds"]

print("Scale lengths:", CONFIG.scale_lengths)
print("Expected folds:", len(folds))
print("Feature files:", len(list(FEATURE_DIR.glob("sub*.npz"))))

if len(list(FEATURE_DIR.glob("sub*.npz"))) != 123:
    raise ValueError(
        "Expected 123 cached subject files. "
        "The uploaded previous output is incomplete."
    )


Scale lengths: (16, 23, 36, 77)
Expected folds: 10
Feature files: 123


In [6]:
# Cell 6: Audit completed folds and clean only incomplete checkpoints

partial_results_path = RESULT_DIR / "fold_results_partial.csv"
partial_history_path = RESULT_DIR / "training_history_partial.csv"
partial_predictions_path = RESULT_DIR / "test_predictions_partial.csv"

if partial_results_path.exists():
    partial_results_df = pd.read_csv(partial_results_path)
    completed_folds = set(
        partial_results_df["fold"].astype(int).tolist()
    )
else:
    partial_results_df = pd.DataFrame()
    completed_folds = set()

checkpoint_files = sorted(
    CHECKPOINT_DIR.glob("fold_*_best.pt")
)

checkpoint_folds = set()

for path in checkpoint_files:
    try:
        checkpoint_folds.add(
            int(path.stem.split("_")[1])
        )
    except Exception:
        pass

incomplete_checkpoint_folds = (
    checkpoint_folds - completed_folds
)

print("Completed folds from partial results:", sorted(completed_folds))
print("Checkpoint folds:", sorted(checkpoint_folds))
print("Incomplete checkpoint folds:", sorted(incomplete_checkpoint_folds))

# Current checkpoints are best checkpoints, not latest-epoch checkpoints.
# Therefore incomplete folds restart cleanly from epoch 1.
CLEAN_INCOMPLETE_CHECKPOINTS = True

if CLEAN_INCOMPLETE_CHECKPOINTS:
    for fold_number in sorted(incomplete_checkpoint_folds):
        path = (
            CHECKPOINT_DIR
            / f"fold_{fold_number:02d}_best.pt"
        )

        if path.exists():
            path.unlink()
            print(
                f"Deleted incomplete Fold {fold_number} checkpoint: "
                f"{path.name}"
            )

remaining_folds = [
    int(fold["fold"])
    for fold in folds
    if int(fold["fold"]) not in completed_folds
]

print("Remaining folds to run:", remaining_folds)


Completed folds from partial results: [1, 2, 3, 4, 5, 6]
Checkpoint folds: [1, 2, 3, 4, 5, 6, 7]
Incomplete checkpoint folds: [7]
Deleted incomplete Fold 7 checkpoint: fold_07_best.pt
Remaining folds to run: [7, 8, 9, 10]


In [7]:
# Cell 7: Configure GPU and deterministic seeds

def set_global_seed(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_global_seed(CONFIG.random_seed)

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU is not enabled. Select a Kaggle T4 GPU and restart."
    )

DEVICE = torch.device("cuda")

test_tensor = torch.tensor(
    [1.0, 2.0, 3.0],
    device=DEVICE,
)
_ = test_tensor.mean()
torch.cuda.synchronize()

print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))
print("Compute capability:", torch.cuda.get_device_capability(0))
print("Selected device:", DEVICE)


PyTorch: 2.10.0+cu128
CUDA: 12.8
GPU: Tesla T4
Compute capability: (7, 5)
Selected device: cuda


In [8]:
# Cell 8: Define lazy rPSD Dataset and DataLoaders

from torch.utils.data import Dataset, DataLoader


class FACEDRPSDDataset(Dataset):
    def __init__(self, subject_ids, feature_dir: Path):
        self.feature_dir = Path(feature_dir)
        self.subject_ids = list(subject_ids)
        self.index = []
        self._cache_subject = None
        self._cache_data = None

        for subject_id in self.subject_ids:
            path = self.feature_dir / f"{subject_id}.npz"

            if not path.exists():
                raise FileNotFoundError(path)

            with np.load(path, allow_pickle=False) as data:
                sample_count = len(data["labels"])

            self.index.extend(
                (subject_id, sample_index)
                for sample_index in range(sample_count)
            )

    def __len__(self):
        return len(self.index)

    def _load_subject(self, subject_id):
        if self._cache_subject != subject_id:
            path = self.feature_dir / f"{subject_id}.npz"

            with np.load(path, allow_pickle=False) as loaded:
                self._cache_data = {
                    f"scale_{length}": loaded[
                        f"scale_{length}"
                    ].copy()
                    for length in CONFIG.scale_lengths
                }
                self._cache_data["labels"] = (
                    loaded["labels"].copy()
                )

            self._cache_subject = subject_id

        return self._cache_data

    def __getitem__(self, index):
        subject_id, sample_index = self.index[index]
        data = self._load_subject(subject_id)

        scales = [
            torch.from_numpy(
                data[f"scale_{length}"][sample_index]
            ).float()
            for length in CONFIG.scale_lengths
        ]

        label = torch.tensor(
            int(data["labels"][sample_index]),
            dtype=torch.long,
        )

        return scales, label, subject_id


def create_fold_loaders(fold, batch_size=None):
    if batch_size is None:
        batch_size = CONFIG.batch_size

    train_dataset = FACEDRPSDDataset(
        fold["train_subjects"],
        FEATURE_DIR,
    )
    val_dataset = FACEDRPSDDataset(
        fold["val_subjects"],
        FEATURE_DIR,
    )
    test_dataset = FACEDRPSDDataset(
        fold["test_subjects"],
        FEATURE_DIR,
    )

    train_generator = torch.Generator()
    train_generator.manual_seed(
        CONFIG.random_seed + int(fold["fold"])
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        drop_last=True,
        num_workers=0,
        pin_memory=True,
        generator=train_generator,
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        drop_last=False,
        num_workers=0,
        pin_memory=True,
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        drop_last=False,
        num_workers=0,
        pin_memory=True,
    )

    return train_loader, val_loader, test_loader


print("Dataset and DataLoaders are ready.")


Dataset and DataLoaders are ready.


In [9]:
# Cell 9: Build and smoke-test the official MSGM model

def build_model(device):
    model = MSGM(
        num_channels=CONFIG.num_channels,
        num_features=CONFIG.num_features,
        num_classes=CONFIG.num_classes,
        hidden_dim=CONFIG.hidden_dim,
        graph_layers=CONFIG.graph_layers,
        chebyshev_order=CONFIG.chebyshev_order,
        scale_lengths=CONFIG.scale_lengths,
        mamba_layers=CONFIG.mamba_layers,
        mamba_state_dim=CONFIG.mamba_state_dim,
        mamba_conv_kernel=CONFIG.mamba_conv_kernel,
        mamba_expand=CONFIG.mamba_expand,
        dropout=CONFIG.dropout,
    )

    return model.to(device)


# Validate one batch from the first remaining fold.
fold_for_test = next(
    fold for fold in folds
    if int(fold["fold"]) not in completed_folds
)

train_loader, _, _ = create_fold_loaders(
    fold_for_test,
    CONFIG.batch_size,
)

batch_scales, batch_labels, _ = next(
    iter(train_loader)
)

model = build_model(DEVICE)
model.eval()

with torch.no_grad():
    smoke_inputs = [
        tensor[:2].contiguous().to(DEVICE)
        for tensor in batch_scales
    ]
    smoke_logits = model(*smoke_inputs)
    torch.cuda.synchronize()

print("Model:", model.__class__.__name__)
print("Trainable parameters:", count_parameters(model))
print("Smoke output shape:", tuple(smoke_logits.shape))

if tuple(smoke_logits.shape) != (2, CONFIG.num_classes):
    raise ValueError(
        f"Unexpected output shape: {tuple(smoke_logits.shape)}"
    )

del model, smoke_inputs, smoke_logits
gc.collect()
torch.cuda.empty_cache()

print("Model smoke test passed.")


Model: MSGM
Trainable parameters: 221480
Smoke output shape: (2, 2)
Model smoke test passed.


In [10]:
# Cell 10: Define training, evaluation, and detailed runtime functions

import gc
import time
import numpy as np
import pandas as pd
import torch

from dataclasses import asdict
from torch import nn
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
)


def synchronize_device():
    """Wait until all pending GPU operations finish before timing."""
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()


def format_seconds(seconds):
    """Return a readable runtime string."""
    seconds = float(seconds)

    if seconds < 60:
        return f"{seconds:.2f} sec"

    minutes = seconds / 60

    if minutes < 60:
        return f"{minutes:.2f} min"

    return f"{minutes / 60:.2f} hr"


def move_scales_to_device(scales):
    """Move all temporal-scale tensors to CPU/GPU."""
    return [
        scale.to(
            DEVICE,
            non_blocking=(DEVICE.type == "cuda"),
        )
        for scale in scales
    ]


def compute_metrics(y_true, y_pred):
    """Calculate accuracy and multiple F1 variants."""

    return {
        "accuracy": accuracy_score(y_true, y_pred),

        "f1_binary": f1_score(
            y_true,
            y_pred,
            average="binary",
            zero_division=0,
        ),

        "f1_macro": f1_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0,
        ),

        "f1_weighted": f1_score(
            y_true,
            y_pred,
            average="weighted",
            zero_division=0,
        ),

        "precision_binary": precision_score(
            y_true,
            y_pred,
            average="binary",
            zero_division=0,
        ),

        "recall_binary": recall_score(
            y_true,
            y_pred,
            average="binary",
            zero_division=0,
        ),
    }


def run_epoch(
    model,
    loader,
    criterion,
    optimizer=None,
):
    """
    Run one complete training, validation, or testing pass.

    Returns:
        metrics
        y_true
        y_pred
        elapsed_seconds
    """

    is_training = optimizer is not None

    if is_training:
        model.train()
    else:
        model.eval()

    total_loss = 0.0
    total_samples = 0

    y_true = []
    y_pred = []

    synchronize_device()
    epoch_start = time.perf_counter()

    for scales, labels, _ in loader:
        scales = move_scales_to_device(scales)

        labels = labels.to(
            DEVICE,
            non_blocking=(DEVICE.type == "cuda"),
        )

        if is_training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(is_training):
            # Four temporal scales are passed separately.
            logits = model(*scales)

            loss = criterion(
                logits,
                labels,
            )

            if is_training:
                loss.backward()
                optimizer.step()

        batch_size = labels.size(0)

        total_loss += loss.item() * batch_size
        total_samples += batch_size

        predictions = logits.argmax(dim=1)

        y_true.extend(
            labels.detach().cpu().numpy().tolist()
        )

        y_pred.extend(
            predictions.detach().cpu().numpy().tolist()
        )

    synchronize_device()
    elapsed_seconds = time.perf_counter() - epoch_start

    if total_samples == 0:
        raise RuntimeError(
            "The DataLoader produced zero samples."
        )

    metrics = compute_metrics(
        y_true,
        y_pred,
    )

    metrics["loss"] = (
        total_loss / total_samples
    )

    metrics["n_samples"] = total_samples
    metrics["runtime_seconds"] = elapsed_seconds
    metrics["runtime_minutes"] = elapsed_seconds / 60

    return (
        metrics,
        np.asarray(y_true),
        np.asarray(y_pred),
        elapsed_seconds,
    )


def train_one_fold(
    fold,
    max_epochs=None,
):
    """
    Train, validate, and test MSGM for one cross-subject fold.

    Timings returned:
        model setup time
        each training epoch time
        each validation epoch time
        total training time
        total validation time
        checkpoint loading time
        test time
        complete fold runtime
    """

    fold_number = int(fold["fold"])

    set_global_seed(
        CONFIG.random_seed + fold_number
    )

    synchronize_device()
    complete_fold_start = time.perf_counter()

    # -------------------------------------------------------
    # DataLoader creation time
    # -------------------------------------------------------
    loader_start = time.perf_counter()

    train_loader, val_loader, test_loader = (
        create_fold_loaders(
            fold,
            batch_size=CONFIG.batch_size,
        )
    )

    loader_setup_seconds = (
        time.perf_counter() - loader_start
    )

    # -------------------------------------------------------
    # Model and optimizer setup time
    # -------------------------------------------------------
    synchronize_device()
    model_setup_start = time.perf_counter()

    model = build_model(DEVICE)

    criterion = nn.CrossEntropyLoss(
        label_smoothing=CONFIG.label_smoothing
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=CONFIG.learning_rate,
        weight_decay=CONFIG.weight_decay,
    )

    synchronize_device()

    model_setup_seconds = (
        time.perf_counter() - model_setup_start
    )

    epochs = (
        CONFIG.epochs
        if max_epochs is None
        else int(max_epochs)
    )

    best_val_accuracy = -np.inf
    best_epoch = -1
    epochs_without_improvement = 0

    checkpoint_path = (
        CHECKPOINT_DIR
        / f"fold_{fold_number:02d}_best.pt"
    )

    if checkpoint_path.exists():
        checkpoint_path.unlink()

    history = []

    total_training_seconds = 0.0
    total_validation_seconds = 0.0
    total_checkpoint_save_seconds = 0.0

    # -------------------------------------------------------
    # Training and validation loop
    # -------------------------------------------------------
    for epoch in range(1, epochs + 1):

        (
            train_metrics,
            _,
            _,
            train_seconds,
        ) = run_epoch(
            model=model,
            loader=train_loader,
            criterion=criterion,
            optimizer=optimizer,
        )

        (
            val_metrics,
            _,
            _,
            val_seconds,
        ) = run_epoch(
            model=model,
            loader=val_loader,
            criterion=criterion,
            optimizer=None,
        )

        total_training_seconds += train_seconds
        total_validation_seconds += val_seconds

        history_row = {
            "fold": fold_number,
            "epoch": epoch,

            "train_time_seconds": train_seconds,
            "train_time_minutes": train_seconds / 60,

            "validation_time_seconds": val_seconds,
            "validation_time_minutes": val_seconds / 60,

            "epoch_total_time_seconds": (
                train_seconds + val_seconds
            ),

            "epoch_total_time_minutes": (
                train_seconds + val_seconds
            ) / 60,

            **{
                f"train_{key}": value
                for key, value
                in train_metrics.items()
            },

            **{
                f"val_{key}": value
                for key, value
                in val_metrics.items()
            },
        }

        history.append(history_row)

        print(
            f"Fold {fold_number:02d} | "
            f"Epoch {epoch:02d}/{epochs:02d}\n"
            f"  Train: "
            f"loss={train_metrics['loss']:.4f}, "
            f"acc={train_metrics['accuracy']:.4f}, "
            f"time={format_seconds(train_seconds)}\n"
            f"  Val:   "
            f"loss={val_metrics['loss']:.4f}, "
            f"acc={val_metrics['accuracy']:.4f}, "
            f"macro-F1={val_metrics['f1_macro']:.4f}, "
            f"time={format_seconds(val_seconds)}\n"
            f"  Epoch total: "
            f"{format_seconds(train_seconds + val_seconds)}"
        )

        if (
            val_metrics["accuracy"]
            > best_val_accuracy
        ):
            best_val_accuracy = float(
                val_metrics["accuracy"]
            )

            best_epoch = epoch
            epochs_without_improvement = 0

            checkpoint = {
                "fold": fold_number,
                "epoch": int(epoch),

                "best_val_accuracy": float(
                    best_val_accuracy
                ),

                "model_state_dict": (
                    model.state_dict()
                ),

                "optimizer_state_dict": (
                    optimizer.state_dict()
                ),

                "val_metrics": {
                    key: (
                        float(value)
                        if isinstance(
                            value,
                            (
                                float,
                                int,
                                np.floating,
                                np.integer,
                            ),
                        )
                        else value
                    )
                    for key, value
                    in val_metrics.items()
                },

                "config": asdict(CONFIG),
            }

            synchronize_device()
            checkpoint_save_start = time.perf_counter()

            torch.save(
                checkpoint,
                checkpoint_path,
            )

            synchronize_device()

            checkpoint_save_seconds = (
                time.perf_counter()
                - checkpoint_save_start
            )

            total_checkpoint_save_seconds += (
                checkpoint_save_seconds
            )

        else:
            epochs_without_improvement += 1

        if (
            epochs_without_improvement
            >= CONFIG.early_stopping_patience
        ):
            print(
                f"\nEarly stopping at epoch {epoch}. "
                f"Best epoch: {best_epoch}"
            )
            break

    if not checkpoint_path.exists():
        raise FileNotFoundError(
            f"No checkpoint was created: "
            f"{checkpoint_path}"
        )

    # -------------------------------------------------------
    # Best checkpoint loading time
    # -------------------------------------------------------
    synchronize_device()
    checkpoint_load_start = time.perf_counter()

    checkpoint = torch.load(
        checkpoint_path,
        map_location=DEVICE,
        weights_only=False,
    )

    model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    synchronize_device()

    checkpoint_load_seconds = (
        time.perf_counter()
        - checkpoint_load_start
    )

    # -------------------------------------------------------
    # Test time
    # -------------------------------------------------------
    (
        test_metrics,
        y_true,
        y_pred,
        test_seconds,
    ) = run_epoch(
        model=model,
        loader=test_loader,
        criterion=criterion,
        optimizer=None,
    )

    synchronize_device()

    complete_fold_seconds = (
        time.perf_counter()
        - complete_fold_start
    )

    history_df = pd.DataFrame(history)

    result = {
        "fold": fold_number,
        "epochs_completed": len(history),
        "best_epoch": int(best_epoch),
        "best_val_accuracy": float(
            best_val_accuracy
        ),

        # Setup timings
        "loader_setup_seconds": (
            loader_setup_seconds
        ),
        "model_setup_seconds": (
            model_setup_seconds
        ),

        # Main timings
        "total_training_seconds": (
            total_training_seconds
        ),
        "total_training_minutes": (
            total_training_seconds / 60
        ),

        "total_validation_seconds": (
            total_validation_seconds
        ),
        "total_validation_minutes": (
            total_validation_seconds / 60
        ),

        "checkpoint_save_seconds": (
            total_checkpoint_save_seconds
        ),

        "checkpoint_load_seconds": (
            checkpoint_load_seconds
        ),

        "test_seconds": test_seconds,
        "test_minutes": test_seconds / 60,

        "complete_fold_seconds": (
            complete_fold_seconds
        ),
        "complete_fold_minutes": (
            complete_fold_seconds / 60
        ),

        **{
            f"test_{key}": value
            for key, value
            in test_metrics.items()
        },
    }

    print("\n" + "=" * 70)
    print(f"Fold {fold_number:02d} timing summary")
    print("=" * 70)

    print(
        "Total training time:   ",
        format_seconds(total_training_seconds),
    )

    print(
        "Total validation time: ",
        format_seconds(total_validation_seconds),
    )

    print(
        "Test time:             ",
        format_seconds(test_seconds),
    )

    print(
        "Checkpoint save time:  ",
        format_seconds(
            total_checkpoint_save_seconds
        ),
    )

    print(
        "Checkpoint load time:  ",
        format_seconds(
            checkpoint_load_seconds
        ),
    )

    print(
        "Complete fold runtime: ",
        format_seconds(
            complete_fold_seconds
        ),
    )

    del model
    del optimizer

    gc.collect()

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    return (
        result,
        history_df,
        y_true,
        y_pred,
    )


print(
    "Cell 30 complete: detailed training, validation, "
    "testing, and total runtime measurement is enabled."
)

Cell 30 complete: detailed training, validation, testing, and total runtime measurement is enabled.


In [11]:
# Cell 11: Resume remaining folds with per-fold saving

RUN_RESUME = True

if not RUN_RESUME:
    print("Resume is disabled.")
else:
    partial_results_path = RESULT_DIR / "fold_results_partial.csv"
    partial_history_path = RESULT_DIR / "training_history_partial.csv"
    partial_predictions_path = RESULT_DIR / "test_predictions_partial.csv"

    final_results_path = RESULT_DIR / "fold_results.csv"
    final_history_path = RESULT_DIR / "training_history.csv"
    final_predictions_path = RESULT_DIR / "test_predictions.csv"

    # Reload completed outputs.
    if partial_results_path.exists():
        existing_results_df = pd.read_csv(partial_results_path)
        existing_results_df["fold"] = (
            existing_results_df["fold"].astype(int)
        )
        completed_folds = set(
            existing_results_df["fold"].tolist()
        )
        fold_results = existing_results_df.to_dict("records")
    else:
        completed_folds = set()
        fold_results = []

    if partial_history_path.exists():
        existing_history_df = pd.read_csv(partial_history_path)
        existing_history_df = existing_history_df[
            existing_history_df["fold"]
            .astype(int)
            .isin(completed_folds)
        ].copy()
        all_histories = [existing_history_df]
    else:
        all_histories = []

    if partial_predictions_path.exists():
        existing_predictions_df = pd.read_csv(
            partial_predictions_path
        )
        existing_predictions_df = existing_predictions_df[
            existing_predictions_df["fold"]
            .astype(int)
            .isin(completed_folds)
        ].copy()
        all_predictions = [existing_predictions_df]
    else:
        all_predictions = []

    print("Completed folds:", sorted(completed_folds))
    print(
        "Remaining folds:",
        [
            int(fold["fold"])
            for fold in folds
            if int(fold["fold"]) not in completed_folds
        ],
    )

    synchronize_device()
    session_start = time.perf_counter()

    observed_fold_times = []

    for fold in folds:
        fold_number = int(fold["fold"])

        if fold_number in completed_folds:
            print(
                f"Skipping Fold {fold_number}: already completed."
            )
            continue

        print("\n" + "=" * 100)
        print(f"Starting Fold {fold_number}/10")
        print("=" * 100)

        synchronize_device()
        fold_start = time.perf_counter()

        result, history_df, y_true, y_pred = (
            train_one_fold(fold)
        )

        synchronize_device()
        fold_wall_seconds = (
            time.perf_counter() - fold_start
        )

        result["fold_wall_seconds"] = fold_wall_seconds
        result["fold_wall_minutes"] = fold_wall_seconds / 60
        result["fold_wall_hours"] = fold_wall_seconds / 3600

        fold_results.append(result)
        all_histories.append(history_df)
        all_predictions.append(
            pd.DataFrame(
                {
                    "fold": fold_number,
                    "y_true": y_true,
                    "y_pred": y_pred,
                }
            )
        )

        completed_folds.add(fold_number)
        observed_fold_times.append(fold_wall_seconds)

        current_results_df = (
            pd.DataFrame(fold_results)
            .sort_values("fold")
            .drop_duplicates(
                subset=["fold"],
                keep="last",
            )
            .reset_index(drop=True)
        )

        current_history_df = (
            pd.concat(
                all_histories,
                ignore_index=True,
            )
            .drop_duplicates(
                subset=["fold", "epoch"],
                keep="last",
            )
            .sort_values(["fold", "epoch"])
            .reset_index(drop=True)
        )

        current_predictions_df = pd.concat(
            all_predictions,
            ignore_index=True,
        )

        current_results_df.to_csv(
            partial_results_path,
            index=False,
        )
        current_history_df.to_csv(
            partial_history_path,
            index=False,
        )
        current_predictions_df.to_csv(
            partial_predictions_path,
            index=False,
        )

        print(f"Fold {fold_number} completed and saved.")
        print(
            "Fold runtime:",
            format_seconds(fold_wall_seconds),
        )
        print(
            "Test accuracy:",
            f"{100 * result['test_accuracy']:.2f}%",
        )
        print(
            "Test macro-F1:",
            f"{100 * result['test_f1_macro']:.2f}%",
        )

        remaining_count = (
            len(folds) - len(completed_folds)
        )

        average_seconds = (
            sum(observed_fold_times)
            / len(observed_fold_times)
        )

        print(
            "Estimated remaining time:",
            format_seconds(
                average_seconds * remaining_count
            ),
        )

        gc.collect()
        torch.cuda.empty_cache()

    session_seconds = time.perf_counter() - session_start

    fold_results_df = (
        pd.DataFrame(fold_results)
        .sort_values("fold")
        .drop_duplicates(
            subset=["fold"],
            keep="last",
        )
        .reset_index(drop=True)
    )

    histories_df = (
        pd.concat(all_histories, ignore_index=True)
        .drop_duplicates(
            subset=["fold", "epoch"],
            keep="last",
        )
        .sort_values(["fold", "epoch"])
        .reset_index(drop=True)
    )

    predictions_df = pd.concat(
        all_predictions,
        ignore_index=True,
    )

    fold_results_df.to_csv(
        final_results_path,
        index=False,
    )
    histories_df.to_csv(
        final_history_path,
        index=False,
    )
    predictions_df.to_csv(
        final_predictions_path,
        index=False,
    )

    print("\nResume run finished.")
    print(
        "Completed folds:",
        fold_results_df["fold"].astype(int).tolist(),
    )
    print(
        "Current session runtime:",
        format_seconds(session_seconds),
    )

    display(fold_results_df)


Completed folds: [1, 2, 3, 4, 5, 6]
Remaining folds: [7, 8, 9, 10]
Skipping Fold 1: already completed.
Skipping Fold 2: already completed.
Skipping Fold 3: already completed.
Skipping Fold 4: already completed.
Skipping Fold 5: already completed.
Skipping Fold 6: already completed.

Starting Fold 7/10
Fold 07 | Epoch 01/30
  Train: loss=0.6921, acc=0.5234, time=10.36 min
  Val:   loss=0.6946, acc=0.5104, macro-F1=0.3997, time=3.28 sec
  Epoch total: 10.42 min
Fold 07 | Epoch 02/30
  Train: loss=0.6871, acc=0.5525, time=10.72 min
  Val:   loss=0.6922, acc=0.5278, macro-F1=0.5073, time=3.46 sec
  Epoch total: 10.77 min
Fold 07 | Epoch 03/30
  Train: loss=0.6814, acc=0.5684, time=10.86 min
  Val:   loss=0.6881, acc=0.5486, macro-F1=0.5364, time=3.28 sec
  Epoch total: 10.92 min
Fold 07 | Epoch 04/30
  Train: loss=0.6729, acc=0.5935, time=10.80 min
  Val:   loss=0.6836, acc=0.5579, macro-F1=0.5473, time=3.39 sec
  Epoch total: 10.86 min
Fold 07 | Epoch 05/30
  Train: loss=0.6628, acc=0.614

,fold,epochs_completed,best_epoch,best_val_accuracy,loader_setup_seconds,model_setup_seconds,total_training_seconds,total_training_minutes,total_validation_seconds,total_validation_minutes,...,test_f1_weighted,test_precision_binary,test_recall_binary,test_loss,test_n_samples,test_runtime_seconds,test_runtime_minutes,fold_wall_seconds,fold_wall_minutes,fold_wall_hours
0,1,11,6,0.605324,0.040869,0.006488,5818.842833,96.980714,28.368078,0.472801,...,0.593478,0.642857,0.458333,0.668220,864,2.418131,0.040302,5850.668725,97.511145,1.625186
1,2,9,4,0.613426,0.037427,0.005536,4506.390898,75.106515,21.264804,0.354413,...,0.591225,0.617143,0.500000,0.674735,864,2.335835,0.038931,4530.829020,75.513817,1.258564
2,3,6,1,0.559028,0.041525,0.005467,3090.259000,51.504317,15.042997,0.250717,...,0.418150,0.641791,0.099537,0.690973,864,2.347028,0.039117,3108.311315,51.805189,0.863420
3,4,21,16,0.636574,0.039521,0.005491,10444.390032,174.073167,49.549870,0.825831,...,0.552083,0.551963,0.553241,0.780354,864,2.340856,0.039014,10497.830448,174.963841,2.916064
4,5,12,7,0.614583,0.040334,0.005609,5881.972660,98.032878,28.146033,0.469101,...,0.542818,0.540541,0.694444,0.689682,864,2.299478,0.038325,5913.411406,98.556857,1.642614
5,6,8,3,0.585648,0.036210,0.005406,3930.303125,65.505052,18.921237,0.315354,...,0.552574,0.547529,0.666667,0.682665,864,2.309990,0.038500,3952.273849,65.871231,1.097854
6,7,25,20,0.646991,0.051590,7.331249,15181.867631,253.031127,76.417558,1.273626,...,0.595047,0.637500,0.472222,0.870652,864,2.885395,0.048090,15270.794072,254.513235,4.241887
7,8,9,4,0.556713,0.054432,0.009483,5278.904616,87.981744,26.060595,0.434343,...,0.597058,0.582868,0.724537,0.672800,864,2.832141,0.047202,5308.807812,88.480130,1.474669
8,9,9,4,0.642361,0.043382,0.006535,5009.216668,83.486944,24.009116,0.400152,...,0.601679,0.606280,0.581019,0.674504,864,2.691058,0.044851,5036.857659,83.947628,1.399127
9,10,25,20,0.594907,0.042551,0.006136,13847.067168,230.784453,66.846005,1.114100,...,0.564815,0.564815,0.564815,0.877646,864,2.655884,0.044265,13918.583516,231.976392,3.866273


In [12]:
# Cell 12: Summarize full cross-validation metrics and runtime

from pathlib import Path
import numpy as np
import pandas as pd


fold_results_path = RESULT_DIR / "fold_results.csv"

if not fold_results_path.exists():
    print(
        "Full fold-results file does not exist yet. "
        "Run Cell 32 first."
    )

else:
    fold_results_df = pd.read_csv(
        fold_results_path
    )

    print(
        f"Loaded {len(fold_results_df)} fold result(s)."
    )

    if len(fold_results_df) < CONFIG.num_test_folds:
        print(
            f"Warning: expected {CONFIG.num_test_folds} folds, "
            f"but found only {len(fold_results_df)}."
        )
        print(
            "This summary is incomplete and should not be "
            "reported as the final 10-fold result."
        )

    # -------------------------------------------------------
    # Classification metrics
    # -------------------------------------------------------
    requested_metric_columns = [
        "test_accuracy",
        "test_f1_binary",
        "test_f1_macro",
        "test_f1_weighted",
        "test_precision_binary",
        "test_recall_binary",
    ]

    metric_columns = [
        column
        for column in requested_metric_columns
        if column in fold_results_df.columns
    ]

    missing_metric_columns = [
        column
        for column in requested_metric_columns
        if column not in fold_results_df.columns
    ]

    if missing_metric_columns:
        print(
            "\nMissing metric columns:",
            missing_metric_columns,
        )

    metric_summary_rows = []

    for column in metric_columns:
        values = pd.to_numeric(
            fold_results_df[column],
            errors="coerce",
        ).dropna()

        if len(values) == 0:
            continue

        mean_value = values.mean()

        std_value = (
            values.std(ddof=1)
            if len(values) > 1
            else np.nan
        )

        metric_summary_rows.append(
            {
                "metric": column,
                "folds_available": len(values),
                "mean": mean_value,
                "std": std_value,
                "mean_percent": 100 * mean_value,
                "std_percent": (
                    100 * std_value
                    if pd.notna(std_value)
                    else np.nan
                ),
                "min_percent": 100 * values.min(),
                "max_percent": 100 * values.max(),
            }
        )

    cv_summary_df = pd.DataFrame(
        metric_summary_rows
    )

    cv_summary_df.to_csv(
        RESULT_DIR / "cv_summary.csv",
        index=False,
    )

    # -------------------------------------------------------
    # Runtime summary
    # -------------------------------------------------------
    requested_runtime_columns = [
        "epochs_completed",
        "total_training_seconds",
        "total_validation_seconds",
        "test_seconds",
        "checkpoint_save_seconds",
        "checkpoint_load_seconds",
        "loader_setup_seconds",
        "model_setup_seconds",
        "complete_fold_seconds",
    ]

    runtime_columns = [
        column
        for column in requested_runtime_columns
        if column in fold_results_df.columns
    ]

    runtime_summary_rows = []

    for column in runtime_columns:
        values = pd.to_numeric(
            fold_results_df[column],
            errors="coerce",
        ).dropna()

        if len(values) == 0:
            continue

        if column == "epochs_completed":
            runtime_summary_rows.append(
                {
                    "measure": column,
                    "folds_available": len(values),
                    "mean": values.mean(),
                    "std": (
                        values.std(ddof=1)
                        if len(values) > 1
                        else np.nan
                    ),
                    "total": values.sum(),
                    "unit": "epochs",
                }
            )

        else:
            runtime_summary_rows.append(
                {
                    "measure": column,
                    "folds_available": len(values),
                    "mean": values.mean(),
                    "std": (
                        values.std(ddof=1)
                        if len(values) > 1
                        else np.nan
                    ),
                    "total": values.sum(),
                    "unit": "seconds",
                }
            )

    runtime_summary_df = pd.DataFrame(
        runtime_summary_rows
    )

    runtime_summary_df.to_csv(
        RESULT_DIR / "runtime_summary.csv",
        index=False,
    )

    # -------------------------------------------------------
    # Human-readable runtime totals
    # -------------------------------------------------------
    total_training_seconds = (
        fold_results_df[
            "total_training_seconds"
        ].sum()
        if "total_training_seconds"
        in fold_results_df.columns
        else np.nan
    )

    total_validation_seconds = (
        fold_results_df[
            "total_validation_seconds"
        ].sum()
        if "total_validation_seconds"
        in fold_results_df.columns
        else np.nan
    )

    total_test_seconds = (
        fold_results_df[
            "test_seconds"
        ].sum()
        if "test_seconds"
        in fold_results_df.columns
        else np.nan
    )

    total_complete_seconds = (
        fold_results_df[
            "complete_fold_seconds"
        ].sum()
        if "complete_fold_seconds"
        in fold_results_df.columns
        else np.nan
    )

    # -------------------------------------------------------
    # Display
    # -------------------------------------------------------
    print("\nPer-fold results:")
    display(fold_results_df)

    print("\nCross-validation metric summary:")
    display(cv_summary_df)

    print("\nRuntime summary:")
    display(runtime_summary_df)

    print("\nOverall runtime totals:")

    if pd.notna(total_training_seconds):
        print(
            "Total training time:",
            format_seconds(
                total_training_seconds
            ),
        )

    if pd.notna(total_validation_seconds):
        print(
            "Total validation time:",
            format_seconds(
                total_validation_seconds
            ),
        )

    if pd.notna(total_test_seconds):
        print(
            "Total test time:",
            format_seconds(
                total_test_seconds
            ),
        )

    if pd.notna(total_complete_seconds):
        print(
            "Complete 10-fold runtime:",
            format_seconds(
                total_complete_seconds
            ),
        )

    # -------------------------------------------------------
    # Reference comparison
    # -------------------------------------------------------
    print("\nPublished MSGM FACED reference target:")
    print("Accuracy: 63.17 ± 3.62%")
    print("Reported F1: 76.01 ± 3.74%")

    print(
        "\nImportant: compare the reported F1 against "
        "binary, macro, and weighted F1 separately because "
        "the exact averaging definition must be verified "
        "before claiming a direct reproduction."
    )

    print("\nSaved files:")
    print(
        RESULT_DIR / "cv_summary.csv"
    )
    print(
        RESULT_DIR / "runtime_summary.csv"
    )

Loaded 10 fold result(s).

Per-fold results:


,fold,epochs_completed,best_epoch,best_val_accuracy,loader_setup_seconds,model_setup_seconds,total_training_seconds,total_training_minutes,total_validation_seconds,total_validation_minutes,...,test_f1_weighted,test_precision_binary,test_recall_binary,test_loss,test_n_samples,test_runtime_seconds,test_runtime_minutes,fold_wall_seconds,fold_wall_minutes,fold_wall_hours
0,1,11,6,0.605324,0.040869,0.006488,5818.842833,96.980714,28.368078,0.472801,...,0.593478,0.642857,0.458333,0.668220,864,2.418131,0.040302,5850.668725,97.511145,1.625186
1,2,9,4,0.613426,0.037427,0.005536,4506.390898,75.106515,21.264804,0.354413,...,0.591225,0.617143,0.500000,0.674735,864,2.335835,0.038931,4530.829020,75.513817,1.258564
2,3,6,1,0.559028,0.041525,0.005467,3090.259000,51.504317,15.042997,0.250717,...,0.418150,0.641791,0.099537,0.690973,864,2.347028,0.039117,3108.311315,51.805189,0.863420
3,4,21,16,0.636574,0.039521,0.005491,10444.390032,174.073167,49.549870,0.825831,...,0.552083,0.551963,0.553241,0.780354,864,2.340856,0.039014,10497.830448,174.963841,2.916064
4,5,12,7,0.614583,0.040334,0.005609,5881.972660,98.032878,28.146033,0.469101,...,0.542818,0.540541,0.694444,0.689682,864,2.299478,0.038325,5913.411406,98.556857,1.642614
5,6,8,3,0.585648,0.036210,0.005406,3930.303125,65.505052,18.921237,0.315354,...,0.552574,0.547529,0.666667,0.682665,864,2.309990,0.038500,3952.273849,65.871231,1.097854
6,7,25,20,0.646991,0.051590,7.331249,15181.867631,253.031127,76.417558,1.273626,...,0.595047,0.637500,0.472222,0.870652,864,2.885395,0.048090,15270.794072,254.513235,4.241887
7,8,9,4,0.556713,0.054432,0.009483,5278.904616,87.981744,26.060595,0.434343,...,0.597058,0.582868,0.724537,0.672800,864,2.832141,0.047202,5308.807812,88.480130,1.474669
8,9,9,4,0.642361,0.043382,0.006535,5009.216668,83.486944,24.009116,0.400152,...,0.601679,0.606280,0.581019,0.674504,864,2.691058,0.044851,5036.857659,83.947628,1.399127
9,10,25,20,0.594907,0.042551,0.006136,13847.067168,230.784453,66.846005,1.114100,...,0.564815,0.564815,0.564815,0.877646,864,2.655884,0.044265,13918.583516,231.976392,3.866273



Cross-validation metric summary:


,metric,folds_available,mean,std,mean_percent,std_percent,min_percent,max_percent
0,test_accuracy,10,0.575231,0.029061,57.523148,2.906094,52.199074,60.300926
1,test_f1_binary,10,0.536844,0.132700,53.684414,13.270013,17.234469,64.602683
2,test_f1_macro,10,0.560893,0.054756,56.089274,5.475642,41.814956,60.167897
3,test_f1_weighted,10,0.560893,0.054756,56.089274,5.475642,41.814956,60.167897
4,test_precision_binary,10,0.593329,0.040832,59.332859,4.083226,54.054054,64.285714
5,test_recall_binary,10,0.531481,0.177191,53.148148,17.719114,9.953704,72.453704



Runtime summary:


,measure,folds_available,mean,std,total,unit
0,epochs_completed,10,13.500000,7.276293,135.000000,epochs
1,total_training_seconds,10,7298.921463,4285.547728,72989.214630,seconds
2,total_validation_seconds,10,35.462629,21.281908,354.626294,seconds
3,test_seconds,10,2.511580,0.230218,25.115795,seconds
4,checkpoint_save_seconds,10,0.092765,0.061006,0.927646,seconds
5,checkpoint_load_seconds,10,0.028822,0.003201,0.288218,seconds
6,loader_setup_seconds,10,0.042784,0.005845,0.427840,seconds
7,model_setup_seconds,10,0.738740,2.316372,7.387401,seconds
8,complete_fold_seconds,10,7338.616107,4308.976518,73386.161065,seconds



Overall runtime totals:
Total training time: 20.27 hr
Total validation time: 5.91 min
Total test time: 25.12 sec
Complete 10-fold runtime: 20.39 hr

Published MSGM FACED reference target:
Accuracy: 63.17 ± 3.62%
Reported F1: 76.01 ± 3.74%

Important: compare the reported F1 against binary, macro, and weighted F1 separately because the exact averaging definition must be verified before claiming a direct reproduction.

Saved files:
/kaggle/working/faced_msgm/results/cv_summary.csv
/kaggle/working/faced_msgm/results/runtime_summary.csv


In [13]:
# Cell 13: Pooled confusion matrix and classification report

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
)

predictions_path = RESULT_DIR / "test_predictions.csv"

if not predictions_path.exists():
    print("Prediction file does not exist yet. Run Cell 11 first.")
else:
    predictions_df = pd.read_csv(predictions_path)

    y_true = predictions_df["y_true"].to_numpy()
    y_pred = predictions_df["y_pred"].to_numpy()

    print("Pooled confusion matrix:")
    print(confusion_matrix(y_true, y_pred))

    print("\nPooled classification report:")
    print(
        classification_report(
            y_true,
            y_pred,
            target_names=["negative", "positive"],
            digits=4,
            zero_division=0,
        )
    )


Pooled confusion matrix:
[[2674 1646]
 [2024 2296]]

Pooled classification report:
              precision    recall  f1-score   support

    negative     0.5692    0.6190    0.5930      4320
    positive     0.5824    0.5315    0.5558      4320

    accuracy                         0.5752      8640
   macro avg     0.5758    0.5752    0.5744      8640
weighted avg     0.5758    0.5752    0.5744      8640



In [14]:
# Cell 14: Package the resumed outputs

archive_base = Path(
    "/kaggle/working/faced_msgm_resumed_outputs"
)

archive_path = shutil.make_archive(
    str(archive_base),
    "zip",
    root_dir=WORK_DIR,
)

print("Created archive:")
print(archive_path)
print(
    "Download it from the Kaggle Output panel "
    "or save the notebook version."
)


Created archive:
/kaggle/working/faced_msgm_resumed_outputs.zip
Download it from the Kaggle Output panel or save the notebook version.
